## Rag App Using Typesense

In [25]:
import typesense 


In [26]:
import os
from dotenv import load_dotenv
load_dotenv()

typesense_api_key = os.getenv('TYPESENSE_API_KEY')

# Corrected config with proper types
client=typesense.Client({
  'nodes': [{
    'host': 'ibfcwqaxldj1to9sp-1.a2.typesense.net',  
    'port': 443, 
    'protocol': 'https'    
  }],
  'api_key':typesense_api_key,
  'connection_timeout_seconds': 2
})

books_schema = {
  'name': 'books',
  'fields': [
    {'name': 'title', 'type': 'string'},
    {'name': 'authors', 'type': 'string[]', 'facet': True},
    {'name': 'publication_year', 'type': 'int32', 'facet': True},
    {'name': 'ratings_count', 'type': 'int32'},
    {'name': 'average_rating', 'type': 'float'}
  ],
  'default_sorting_field': 'ratings_count'
}
try:
    result = client.collections.create(books_schema)
    print("Collection created successfully:", result)
except Exception as e:
    print(f"Error creating collection: {e}")


Error creating collection: [Errno 409] A collection with name `books` already exists.


In [27]:
client

In [28]:
with open('books.jsonl', 'r', encoding='utf-8') as jsonl_file:
  data = jsonl_file.read()
  client.collections['books'].documents.import_(data)

In [29]:
search_paramaters={
  'q':"harry potter",
  'query_by': "title,authors",
  'sort_by': "ratings_count:desc"
}

client.collections['books'].documents.search(search_paramaters)

{'facet_counts': [],
 'found': 17,
 'hits': [{'document': {'authors': ['J.K. Rowling', ' Mary GrandPré'],
    'average_rating': 4.44,
    'id': '2',
    'image_url': 'https://images.gr-assets.com/books/1474154022m/3.jpg',
    'publication_year': 1997,
    'ratings_count': 4602479,
    'title': "Harry Potter and the Philosopher's Stone"},
   'highlight': {'title': {'matched_tokens': ['Harry', 'Potter'],
     'snippet': "<mark>Harry</mark> <mark>Potter</mark> and the Philosopher's Stone"}},
   'highlights': [{'field': 'title',
     'matched_tokens': ['Harry', 'Potter'],
     'snippet': "<mark>Harry</mark> <mark>Potter</mark> and the Philosopher's Stone"}],
   'text_match': 1157451471441102969,
   'text_match_info': {'best_field_score': '2211897868289',
    'best_field_weight': 15,
    'fields_matched': 1,
    'num_tokens_dropped': 0,
    'score': '1157451471441102969',
    'tokens_matched': 2,
    'typo_prefix_score': 0}},
  {'document': {'authors': ['J.K. Rowling', ' Mary GrandPré', ' R

In [30]:
search_paramaters={
  'q'         :"harry potter",
  'query_by'  : "title",
  'filter_by' : "publication_year:<1998",
  'sort_by'   : "ratings_count:desc"
}

client.collections['books'].documents.search(search_paramaters)

{'facet_counts': [],
 'found': 1,
 'hits': [{'document': {'authors': ['J.K. Rowling', ' Mary GrandPré'],
    'average_rating': 4.44,
    'id': '2',
    'image_url': 'https://images.gr-assets.com/books/1474154022m/3.jpg',
    'publication_year': 1997,
    'ratings_count': 4602479,
    'title': "Harry Potter and the Philosopher's Stone"},
   'highlight': {'title': {'matched_tokens': ['Harry', 'Potter'],
     'snippet': "<mark>Harry</mark> <mark>Potter</mark> and the Philosopher's Stone"}},
   'highlights': [{'field': 'title',
     'matched_tokens': ['Harry', 'Potter'],
     'snippet': "<mark>Harry</mark> <mark>Potter</mark> and the Philosopher's Stone"}],
   'text_match': 1157451471441102969,
   'text_match_info': {'best_field_score': '2211897868289',
    'best_field_weight': 15,
    'fields_matched': 1,
    'num_tokens_dropped': 0,
    'score': '1157451471441102969',
    'tokens_matched': 2,
    'typo_prefix_score': 0}}],
 'out_of': 9979,
 'page': 1,
 'request_params': {'collection_name

In [31]:
search_paramaters={
  'q'         : "experyment",
  'query_by'  : "title",
  'facet_by'  : "authors",
  'sort_by'   : "average_rating:desc"
}

client.collections['books'].documents.search(search_paramaters)

{'facet_counts': [{'counts': [{'count': 1,
     'highlighted': ' Käthe Mazur',
     'value': ' Käthe Mazur'},
    {'count': 1, 'highlighted': 'Mahatma Gandhi', 'value': 'Mahatma Gandhi'},
    {'count': 1, 'highlighted': 'Gretchen Rubin', 'value': 'Gretchen Rubin'},
    {'count': 1,
     'highlighted': 'James Patterson',
     'value': 'James Patterson'}],
   'field_name': 'authors',
   'sampled': False,
   'stats': {'total_values': 4}}],
 'found': 3,
 'hits': [{'document': {'authors': ['James Patterson'],
    'average_rating': 4.08,
    'id': '569',
    'image_url': 'https://images.gr-assets.com/books/1339277875m/13152.jpg',
    'publication_year': 2005,
    'ratings_count': 172302,
    'title': 'The Angel Experiment'},
   'highlight': {'title': {'matched_tokens': ['Experiment'],
     'snippet': 'The Angel <mark>Experiment</mark>'}},
   'highlights': [{'field': 'title',
     'matched_tokens': ['Experiment'],
     'snippet': 'The Angel <mark>Experiment</mark>'}],
   'text_match': 5787300

In [32]:
### langchain + Typesense + Groq LLM + RAG Application

from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Typesense
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq

In [33]:
#initialize the groq LLm (Set GROQ_API_KEY in env)
groq_api_key = os.getenv('GROQ_API_KEY')

In [43]:
loader = TextLoader("test.txt", encoding="utf-8") 
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
docs = text_splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5037.40it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [44]:
docsearch = Typesense.from_documents(
  docs,
  embeddings,
  typesense_client_params={
        "host": "ibfcwqaxldj1to9sp-1.a2.typesense.net",  # Use xxx.a1.typesense.net for Typesense Cloud
        "port": "443",  # Use 443 for Typesense Cloud
        "protocol": "https",  # Use https for Typesense Cloud
        "typesense_api_key":typesense_api_key,
        "typesense_collection_name": "lang-chain"
    },
)

In [46]:
query = "What is artificial intellegence"
found_docs = docsearch.similarity_search(query)
print(found_docs[0].page_content)

Artificial Intelligence (AI)

Artificial Intelligence (AI) is one of the most transformative technologies of the 21st century. At its core, AI refers to the development of computer systems that can perform tasks traditionally requiring human intelligence, such as reasoning, problem-solving, learning, and decision-making. While the concept of machines that think dates back to ancient myths and early philosophical discussions, the formal study of AI began in the mid-20th century, with pioneers like Alan Turing and John McCarthy laying the foundation for modern research.


In [47]:
### Retriever
retriever = docsearch.as_retriever()
retriever

VectorStoreRetriever(tags=['Typesense', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.typesense.Typesense object at 0x000001CB8F4F2660>, search_kwargs={})

In [48]:
query = "Artificial intelligence indepth explanation"
retriever.invoke(query)[0]

Document(metadata={'source': 'test.txt'}, page_content='Artificial Intelligence (AI)\n\nArtificial Intelligence (AI) is one of the most transformative technologies of the 21st century. At its core, AI refers to the development of computer systems that can perform tasks traditionally requiring human intelligence, such as reasoning, problem-solving, learning, and decision-making. While the concept of machines that think dates back to ancient myths and early philosophical discussions, the formal study of AI began in the mid-20th century, with pioneers like Alan Turing and John McCarthy laying the foundation for modern research.')